# Uji LIME Asli via Skrip Pembuktian Langsung 

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoConfig, AutoModel
from lime.lime_text import LimeTextExplainer
import numpy as np
import time

print("=" * 70)
print("🔬 PENGUJIAN LIME ASLI (BERBASIS FORWARD-PASS MODEL INDOBERT)")
print("=" * 70)

# 1. SETUP MODEL & TOKENIZER DARI CHECKPOINT LOKAL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "../saved_models/indobert_focal_loss/best_model.pt"
model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

class IndoBERTClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.indobert = AutoModel.from_config(config)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        out = self.indobert(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(out.last_hidden_state[:, 0, :]))

model = IndoBERTClassifier().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f"✅ Model IndoBERT ({model_path}) berhasil dimuat ke {device}!\n")

# 2. DEFINISI PREDICT_PROBA UNTUK LIME (BATCH INFERENCE)
def predict_proba_lime(texts):
    probs_list = []
    batch_size = 32
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]
        enc = tokenizer(batch, max_length=128, padding='max_length', truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(enc['input_ids'], enc['attention_mask'])
            probs = F.softmax(logits, dim=1).cpu().numpy()
            probs_list.extend(probs)
    return np.array(probs_list)

# 3. INSIALISASI LIME ASLI
explainer = LimeTextExplainer(class_names=["FAKTA", "HOAKS"], random_state=42)

# 4. UJI DENGAN NASKAH HOAKS POLITIK NYATA
test_sentence = "Beredar video viral di media sosial bahwa KPU curang mengunci perolehan suara pemilu tadi malam secara diam-diam."

print(f"Naskah Uji Coba:\n\"{test_sentence}\"\n")
print("--> Menjalankan 100 Perturbasi Teks ke IndoBERT (Tunggu ~2 detik)...")

t0 = time.time()
exp = explainer.explain_instance(test_sentence, predict_proba_lime, num_features=8, num_samples=100)
durasi = time.time() - t0

# 5. CETAK HASIL PEMBUKTIAN MATEMATIS LIME
print(f"⏱️ Waktu Komputasi LIME Riil: {durasi:.2f} detik (Membuktikan inferensi 100x berjalan!)\n")
print("=" * 70)
print(f"{'KATA / TOKEN':<20} | {'BOBOT LIME ASLI':<18} | {'PENGARUH KE KEPUTUSAN'}")
print("=" * 70)

for word, weight in exp.as_list(label=1):
    arah = "🔴 Mendorong ke HOAKS" if weight > 0 else "🟢 Mendorong ke FAKTA"
    # Cetak bobot asli desimal
    print(f"{word:<20} | {weight:>+10.4f}         | {arah}")

print("=" * 70)

🔬 PENGUJIAN LIME ASLI (BERBASIS FORWARD-PASS MODEL INDOBERT)
✅ Model IndoBERT (../saved_models/indobert_focal_loss/best_model.pt) berhasil dimuat ke cpu!

Naskah Uji Coba:
"Beredar video viral di media sosial bahwa KPU curang mengunci perolehan suara pemilu tadi malam secara diam-diam."

--> Menjalankan 100 Perturbasi Teks ke IndoBERT (Tunggu ~2 detik)...
⏱️ Waktu Komputasi LIME Riil: 8.23 detik (Membuktikan inferensi 100x berjalan!)

KATA / TOKEN         | BOBOT LIME ASLI    | PENGARUH KE KEPUTUSAN
video                |    +0.0026         | 🔴 Mendorong ke HOAKS
media                |    +0.0021         | 🔴 Mendorong ke HOAKS
viral                |    +0.0020         | 🔴 Mendorong ke HOAKS
curang               |    +0.0020         | 🔴 Mendorong ke HOAKS
Beredar              |    +0.0016         | 🔴 Mendorong ke HOAKS
mengunci             |    -0.0014         | 🟢 Mendorong ke FAKTA
tadi                 |    +0.0012         | 🔴 Mendorong ke HOAKS
di                   |    -0.0011       